In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
model_path = Path("/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/calibration/gmr_dayblock_final_pm10.pkl")

with open(model_path, "rb") as f:
    gmm_loaded = pickle.load(f)

print(f"Loaded model from: {model_path}")

Loaded model from: /Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/calibration/gmr_dayblock_final_pm10.pkl


In [3]:
# ---- PM10 ----
pm10_files = [
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/pm/summarized/pm10_community_hourly_20231024-20240816.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/pm/summarized/pm10_community_hourly_20240816-20241031.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/pm/summarized/pm10_community_hourly_20241031-20250901.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/pm/summarized/pm10_community_hourly_20250831-20260315.csv"
]

pm10 = pd.concat([pd.read_csv(f) for f in pm10_files], ignore_index=True)


# ---- Temperature ----
temp_files = [
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/weather/summarized/temp_hourly_20230815-20240820.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/weather/summarized/temp_hourly_20240816-20241031.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/weather/summarized/temp_hourly_20241031-20250901.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/weather/summarized/temp_hourly_20250831-20260315.csv"
]

temp = pd.concat([pd.read_csv(f) for f in temp_files], ignore_index=True)


# ---- RH ----
rh_files = [
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/weather/summarized/rh_hourly_20230815-20240820.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/weather/summarized/rh_hourly_20240816-20241031.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/weather/summarized/rh_hourly_20241031-20250901.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/weather/summarized/rh_hourly_20250831-20260315.csv"
]

rh = pd.concat([pd.read_csv(f) for f in rh_files], ignore_index=True)

pm10 = pm10.drop_duplicates()
temp = temp.drop_duplicates()
rh   = rh.drop_duplicates()

In [4]:
# keep non-missing measurements
pm10 = pm10.loc[pm10["mean_pm10"].notna()].copy()
temp = temp.loc[temp["mean_met_temp"].notna()].copy()
rh   = rh.loc[rh["mean_met_rh"].notna()].copy()

# drop extra columns if present
for df, cols in [
    (pm10, ["n_minute_obs", "n_active", "fleet_average_pm10"]),
    (temp, ["n_minute_obs", "n_active", "fleet_average_met_temp"]),
    (rh,   ["n_minute_obs", "n_active", "fleet_average_met_rh"]),
]:
    drop_cols = [c for c in cols if c in df.columns]
    df.drop(columns=drop_cols, inplace=True)

# merge
modulair_apply = (
    pm10.merge(temp, on=["monitor", "date", "hour"], how="outer")
        .merge(rh, on=["monitor", "date", "hour"], how="outer")
)

# rename columns to match model inputs
modulair_apply = modulair_apply.rename(columns={
    "mean_pm10": "mod_pm10",
    "mean_met_temp": "mod_temp",
    "mean_met_rh": "mod_rh"
})

In [5]:
# parse date
modulair_apply["date"] = pd.to_datetime(modulair_apply["date"])

# make datetime if useful
modulair_apply["date_hour"] = pd.to_datetime(
    modulair_apply["date"].dt.strftime("%Y-%m-%d") + " " +
    modulair_apply["hour"].astype(int).astype(str) + ":00:00",
    utc=True
)

# cyclic hour variables
modulair_apply["sin_hour"] = np.sin(2 * np.pi * modulair_apply["hour"] / 24)
modulair_apply["cos_hour"] = np.cos(2 * np.pi * modulair_apply["hour"] / 24)

# season variable: Harmattan = Dec-Feb
modulair_apply["month"] = modulair_apply["date"].dt.month
modulair_apply["season_binary"] = np.where(modulair_apply["month"].isin([12, 1, 2]), 1, 0)

In [6]:
x_cols = ["mod_pm10", "mod_temp", "mod_rh", "sin_hour", "cos_hour", "season_binary"]

modulair_apply_valid = modulair_apply.dropna(subset=x_cols).copy()

print("Rows available for calibration:", len(modulair_apply_valid))

Rows available for calibration: 448880


In [7]:
X_full = modulair_apply_valid[x_cols].to_numpy()

Y_full = gmm_loaded.predict(
    np.array([0, 1, 2, 3, 4, 5]),
    X_full
).ravel()

modulair_apply_valid["pm10_fem_gmr"] = np.maximum(Y_full, 0)
modulair_apply_valid["delta_gmr"] = modulair_apply_valid["pm10_fem_gmr"] - modulair_apply_valid["mod_pm10"]
modulair_apply_valid["ratio_gmr"] = np.where(
    modulair_apply_valid["mod_pm10"] > 0,
    modulair_apply_valid["pm10_fem_gmr"] / modulair_apply_valid["mod_pm10"],
    np.nan
)

modulair_apply_valid.head()

,monitor,date,hour,mod_pm10,mod_temp,mod_rh,date_hour,sin_hour,cos_hour,month,season_binary,pm10_fem_gmr,delta_gmr,ratio_gmr
0,MOD-00077,2024-09-03,16.0,67.721167,35.835000,46.395000,2024-09-03 16:00:00+00:00,-0.866025,-5.000000e-01,9,0,37.996895,-29.724272,0.561079
1,MOD-00077,2024-09-03,17.0,104.549500,32.908333,53.998333,2024-09-03 17:00:00+00:00,-0.965926,-2.588190e-01,9,0,25.977838,-78.571662,0.248474
2,MOD-00077,2024-09-03,18.0,157.049500,30.603333,62.325000,2024-09-03 18:00:00+00:00,-1.000000,-1.836970e-16,9,0,26.688252,-130.361248,0.169935
3,MOD-00077,2024-09-03,19.0,155.377500,29.363333,68.226667,2024-09-03 19:00:00+00:00,-0.965926,2.588190e-01,9,0,26.709456,-128.668044,0.171900
4,MOD-00077,2024-09-03,20.0,86.096167,28.330000,72.103333,2024-09-03 20:00:00+00:00,-0.866025,5.000000e-01,9,0,24.199803,-61.896364,0.281079


In [8]:
pred_cols = ["pm10_fem_gmr", "delta_gmr", "ratio_gmr"]

modulair_apply.loc[modulair_apply_valid.index, pred_cols] = modulair_apply_valid[pred_cols]

modulair_apply

,monitor,date,hour,mod_pm10,mod_temp,mod_rh,date_hour,sin_hour,cos_hour,month,season_binary,pm10_fem_gmr,delta_gmr,ratio_gmr
0,MOD-00077,2024-09-03,16.0,67.721167,35.835000,46.395000,2024-09-03 16:00:00+00:00,-0.866025,-5.000000e-01,9,0,37.996895,-29.724272,0.561079
1,MOD-00077,2024-09-03,17.0,104.549500,32.908333,53.998333,2024-09-03 17:00:00+00:00,-0.965926,-2.588190e-01,9,0,25.977838,-78.571662,0.248474
2,MOD-00077,2024-09-03,18.0,157.049500,30.603333,62.325000,2024-09-03 18:00:00+00:00,-1.000000,-1.836970e-16,9,0,26.688252,-130.361248,0.169935
3,MOD-00077,2024-09-03,19.0,155.377500,29.363333,68.226667,2024-09-03 19:00:00+00:00,-0.965926,2.588190e-01,9,0,26.709456,-128.668044,0.171900
4,MOD-00077,2024-09-03,20.0,86.096167,28.330000,72.103333,2024-09-03 20:00:00+00:00,-0.866025,5.000000e-01,9,0,24.199803,-61.896364,0.281079
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
697202,MOD-PM-01093,2026-03-12,5.0,23.980893,27.834821,71.684643,2026-03-12 05:00:00+00:00,0.965926,2.588190e-01,3,0,21.741082,-2.239811,0.906600
697203,MOD-PM-01093,2026-03-13,7.0,29.580189,30.366604,65.292075,2026-03-13 07:00:00+00:00,0.965926,-2.588190e-01,3,0,29.143196,-0.436993,0.985227
697204,MOD-PM-01093,2026-03-13,8.0,31.163981,32.395357,59.195893,2026-03-13 08:00:00+00:00,0.866025,-5.000000e-01,3,0,31.030135,-0.133846,0.995705
697205,MOD-PM-01093,2026-03-13,9.0,31.205508,34.749322,52.739661,2026-03-13 09:00:00+00:00,0.707107,-7.071068e-01,3,0,31.830713,0.625204,1.020035


In [9]:
outpath = "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/pm/GRIMM_calibrated_pm_summaries/hourly_pm10_modulair_full_calibrated_gmr_20231024-20260315.csv"
modulair_apply.to_csv(outpath, index=False)

print(f"Saved calibrated data to: {outpath}")

Saved calibrated data to: /Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/pm/GRIMM_calibrated_pm_summaries/hourly_pm10_modulair_full_calibrated_gmr_20231024-20260315.csv


In [10]:

# make sure date is datetime
modulair_apply["date"] = pd.to_datetime(modulair_apply["date"])

daily_summary = (
    modulair_apply
    .groupby(["monitor", "date"], as_index=False)
    .agg(
        n_complete_hours_raw=("mod_pm10", lambda x: x.notna().sum()),
        mean_mod_pm10=("mod_pm10", lambda x: x.mean(skipna=True)),
        
        n_complete_hours_gmr=("pm10_fem_gmr", lambda x: x.notna().sum()),
        mean_pm10_fem_gmr=("pm10_fem_gmr", lambda x: x.mean(skipna=True))
    )
)

# apply >=18 hour rule separately to raw and calibrated summaries
daily_summary["mean_mod_pm10"] = np.where(
    daily_summary["n_complete_hours_raw"] >= 18,
    daily_summary["mean_mod_pm10"],
    np.nan
)

daily_summary["mean_pm10_fem_gmr"] = np.where(
    daily_summary["n_complete_hours_gmr"] >= 18,
    daily_summary["mean_pm10_fem_gmr"],
    np.nan
)

# optional comparison columns
daily_summary["delta_daily_gmr"] = (
    daily_summary["mean_pm10_fem_gmr"] - daily_summary["mean_mod_pm10"]
)

daily_summary["ratio_daily_gmr"] = np.where(
    daily_summary["mean_mod_pm10"] > 0,
    daily_summary["mean_pm10_fem_gmr"] / daily_summary["mean_mod_pm10"],
    np.nan
)

daily_summary.head()

,monitor,date,n_complete_hours_raw,mean_mod_pm10,n_complete_hours_gmr,mean_pm10_fem_gmr,delta_daily_gmr,ratio_daily_gmr
0,MOD-00077,2024-09-03,8,NaN,8,NaN,NaN,NaN
1,MOD-00077,2024-09-04,24,46.671007,24,23.064450,-23.606557,0.494192
2,MOD-00077,2024-09-05,24,47.574569,24,23.056567,-24.518003,0.484641
3,MOD-00077,2024-09-06,21,48.806580,21,27.717570,-21.089009,0.567906
4,MOD-00077,2024-09-07,24,61.690326,24,21.384900,-40.305427,0.346649


In [11]:
daily_outpath = "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/pm/GRIMM_calibrated_pm_summaries/daily_pm10_modulair_full_calibrated_gmr_20231024-20260315.csv"
daily_summary.to_csv(daily_outpath, index=False)

print(f"Saved daily summary to: {daily_outpath}")

Saved daily summary to: /Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/pm/GRIMM_calibrated_pm_summaries/daily_pm10_modulair_full_calibrated_gmr_20231024-20260315.csv
